# Practical P30: CLI Tool: AI Text Classifier
**Learning Outcome**: Build a batch AI classification CLI tool with explanation mode.

## Part 1: Creating `ai_classify.py`
Let's write a python CLI tool that parses customer tickets CSV, classifies them into categories, and optionally provides reasons.


In [ ]:
%%writefile ai_classify.py
import argparse
import csv
import json
import time
import sys

def mock_classify(text, categories, explain=False):
    # Simple regex category mapper for mock testing
    text_lower = text.lower()
    matched = 'General'
    for cat in categories:
        if cat.lower() in text_lower or (cat.lower() == 'technical' and ('password' in text_lower or 'app' in text_lower)):
            matched = cat
            break
            
    reason = f"Contains indicators matching the category '{matched}'." if explain else None
    return matched, reason

def main():
    parser = argparse.ArgumentParser(description='AI Text Classifier CLI')
    parser.add_argument('--file', required=True, help='Path to CSV input file')
    parser.add_argument('--categories', required=True, help='Comma separated categories list')
    parser.add_argument('--explain', action='store_true', help='Include reasons')
    
    args = parser.parse_args()
    cats = [c.strip() for c in args.categories.split(',')]
    
    try:
        with open(args.file, 'r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            rows = list(reader)
    except FileNotFoundError:
        print(f'Error: File {args.file} not found.')
        sys.exit(1)
        
    results = []
    for row in rows:
        label, reason = mock_classify(row['text'], cats, args.explain)
        res_row = {
            'ticket_id': row['ticket_id'],
            'text': row['text'],
            'predicted_category': label
        }
        if args.explain:
            res_row['explanation'] = reason
        results.append(res_row)
        
    # Save back to CSV output
    out_path = 'data/classification_results.csv'
    fieldnames = ['ticket_id', 'text', 'predicted_category']
    if args.explain:
        fieldnames.append('explanation')
        
    with open(out_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
        
    print(f'Successfully classified {len(results)} rows. Saved results to {out_path}')

if __name__ == '__main__':
    main()


## Part 2: Running the Classification CLI tool
Let's execute classification command with billing/technical/general categories list.


In [ ]:
!python3 ai_classify.py --file data/tickets.csv --categories "Billing, Technical, Complaint, General" --explain


In [ ]:
# Inspect generated csv output
import pandas as pd
pd.read_csv('data/classification_results.csv')
